# 4-2. **Named Entity Recognition**

The subtask of information extraction that seeks to locate and classify named entity mentions in unstructured text into pre-defined categories such as the person names, organizations, locations, medical codes, time expressions, quantities, monetary values, percentages, etc.

It is very common that sequence modeling, such as HMM, MEMM, CRF, is applied to named entity prediction. In this lab, we will train and test the named entity prediction with sequence modeling, such as CRF.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [2]:
!wget https://raw.githubusercontent.com/kimtwan/NLP_lecture/master/data/ner_dataset.csv

--2026-05-10 02:08:31--  https://raw.githubusercontent.com/kimtwan/NLP_lecture/master/data/ner_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15208261 (15M) [text/plain]
Saving to: ‘ner_dataset.csv.1’

ner_dataset.csv.1   100%[===================>]  14.50M  --.-KB/s    in 0.1s    

2026-05-10 02:08:32 (128 MB/s) - ‘ner_dataset.csv.1’ saved [15208261/15208261]



In [3]:
# read IOB tagged NER dataset as dataframe
df = pd.read_csv('ner_dataset.csv', encoding = 'ISO-8859-1')
df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


In the data, you can see the different types of entities:
* geo = Geographical Entity
* org = Organization
* per = Person
* gpe = Geopolitical Entity
* tim = Time indicator
* art = Artifact
* eve = Event
* nat = Natural Phenomenon

## Data Preprocessing
There are too many NaN values in ‘Sentence #” column, fill NaN by preceding values.
We have 47595 sentences that contain 35172 unique words and tagged by 17 tags.

In [4]:
df = df.ffill()
df['Sentence #'].nunique(), df.Word.nunique(), df.Tag.nunique()

(47959, 35171, 17)

In [5]:
df.groupby('Tag').size().reset_index(name='counts')

,Tag,counts
0,B-art,402
1,B-eve,308
2,B-geo,37644
3,B-gpe,15870
4,B-nat,201
5,B-org,20143
6,B-per,16990
7,B-tim,20333
8,I-art,297
9,I-eve,253


We will now train a CRF model for named entity recognition using sklearn-crfsuite on our dataset. As mentioned before, MEMM or CRF is often used for labeling or parsing of sequential data for named entity recognition.

In [6]:
!pip install -q -U sklearn_crfsuite

##Conditional random fields (CRF)

In [7]:
import sklearn_crfsuite
from sklearn_crfsuite import scorers
from sklearn_crfsuite.utils import flatten
from collections import Counter

In [8]:
agg_func = lambda s: [(w, p, t) for w, p, t in zip(s['Word'].values.tolist(),
                                                   s['POS'].values.tolist(),
                                                   s['Tag'].values.tolist())]
df_grp = df.groupby('Sentence #').apply(agg_func, include_groups=False)
sentences = [s for s in df_grp]

We extract more features (word parts, simplified POS tags, lower/title/upper flags, features of nearby words) and convert them to sklearn-crfsuite format — each sentence should be converted to a list of dicts. The following code were taken from [sklearn-crfsuites official site](https://sklearn-crfsuite.readthedocs.io/en/latest/tutorial.html).

In [9]:
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
    }
    if i > 0:
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })
    else:
        features['EOS'] = True

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, postag, label in sent]

In [10]:
# data splitting for training and testing
X = [sent2features(s) for s in sentences]
y = [sent2labels(s) for s in sentences]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0)

In [11]:
# train a CRF model for named entity recognition using sklearn-crfsuite
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
# to prevent 'keep_tempfiles' AttributeError in scikit-learn >= 0.24
try:
    crf.fit(X_train, y_train) # This will take about 3 mins
except AttributeError:
    pass

In [12]:
# predict
y_pred = crf.predict(X_test)

Because tag “O” (outside) is the most common tag and it will make our results look much better than they actual are. So we remove tag “O” when we evaluate classification metrics.

In [13]:
y = df.Tag.values
classes = np.unique(y)
classes = classes.tolist()
classes.pop() # pop the last item, which is 'O'
classes

['B-art',
 'B-eve',
 'B-geo',
 'B-gpe',
 'B-nat',
 'B-org',
 'B-per',
 'B-tim',
 'I-art',
 'I-eve',
 'I-geo',
 'I-gpe',
 'I-nat',
 'I-org',
 'I-per',
 'I-tim']

In [14]:
#evaluation
y_test_flat = flatten(y_test)
y_pred_flat = flatten(y_pred)
print(classification_report(y_test_flat, y_pred_flat, labels=classes))

              precision    recall  f1-score   support

       B-art       0.46      0.13      0.21       143
       B-eve       0.55      0.40      0.46       106
       B-geo       0.86      0.91      0.88     12447
       B-gpe       0.97      0.94      0.95      5284
       B-nat       0.80      0.42      0.55        78
       B-org       0.80      0.73      0.77      6615
       B-per       0.85      0.83      0.84      5652
       B-tim       0.93      0.88      0.90      6856
       I-art       0.14      0.04      0.06       105
       I-eve       0.36      0.23      0.28        93
       I-geo       0.82      0.80      0.81      2520
       I-gpe       0.88      0.61      0.72        69
       I-nat       1.00      0.39      0.56        23
       I-org       0.82      0.80      0.81      5597
       I-per       0.85      0.90      0.87      5674
       I-tim       0.84      0.75      0.79      2207

   micro avg       0.86      0.85      0.85     53469
   macro avg       0.75   

The following shows what our classifier learned. It is very likely that the beginning of a geographical entity (B-geo) will be followed by a token inside geographical entity (I-geo), but transitions to inside of an organization name (I-org) from tokens with other labels are penalized hugely.

In [15]:
# use the dictionary like a count list to get the sorted result
Counter(crf.transition_features_).most_common(10)

[(('I-art', 'I-art'), 6.47999),
 (('B-art', 'I-art'), 6.394846),
 (('B-nat', 'I-nat'), 6.118064),
 (('I-eve', 'I-eve'), 5.972066),
 (('B-eve', 'I-eve'), 5.743972),
 (('I-tim', 'I-tim'), 5.199972),
 (('I-gpe', 'I-gpe'), 4.896438),
 (('B-tim', 'I-tim'), 4.820086),
 (('I-org', 'I-org'), 4.796007),
 (('B-org', 'I-org'), 4.466231)]

In [16]:
def print_transitions(trans_features):
    for (label_from, label_to), weight in trans_features:
        print('%-6s -> %-7s %0.6f' % (label_from, label_to, weight))
print('Top likely transitions:')
print_transitions(Counter(crf.transition_features_).most_common(20))
print('\nTop unlikely transitions:')
print_transitions(Counter(crf.transition_features_).most_common()[-20:])

Top likely transitions:
I-art  -> I-art   6.479990
B-art  -> I-art   6.394846
B-nat  -> I-nat   6.118064
I-eve  -> I-eve   5.972066
B-eve  -> I-eve   5.743972
I-tim  -> I-tim   5.199972
I-gpe  -> I-gpe   4.896438
B-tim  -> I-tim   4.820086
I-org  -> I-org   4.796007
B-org  -> I-org   4.466231
B-gpe  -> I-gpe   4.029524
B-per  -> I-per   3.806519
O      -> O       3.462313
B-geo  -> I-geo   3.350624
I-per  -> I-per   3.159554
I-geo  -> I-geo   3.044153
I-nat  -> I-nat   2.942678
I-geo  -> B-art   1.935521
O      -> B-per   1.507156
O      -> B-tim   1.501103

Top unlikely transitions:
B-per  -> I-org   -4.219536
B-geo  -> B-geo   -4.280767
B-tim  -> B-tim   -4.348527
B-per  -> I-geo   -4.361884
I-org  -> I-per   -4.368630
I-org  -> I-geo   -4.462026
B-geo  -> I-gpe   -4.557670
B-org  -> I-geo   -4.606129
B-geo  -> I-per   -4.816110
B-org  -> I-per   -4.849032
B-geo  -> I-org   -5.037208
B-gpe  -> I-org   -5.056551
B-gpe  -> I-geo   -5.078297
B-gpe  -> B-gpe   -5.280733
I-per  -> B-per  

##Named Entity Recognition with Spacy

The following lines show how to build named entity recognizer with [SpaCy](https://spacy.io/), to identify the names of things, such as persons, organizations, or locations. SpaCy’s named entity recognition has been trained on the [OntoNotes 5 corpus](https://catalog.ldc.upenn.edu/LDC2013T19) and it supports the following entity types: https://spacy.io/api/annotation#section-named-entities

In [17]:
import spacy

In [18]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 83.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
import spacy
from spacy import displacy
from collections import Counter
import en_core_web_sm

In [20]:
# loading pre-trained model of NER
nlp = en_core_web_sm.load()

In [21]:
!pip install -q wikipedia
import wikipedia

We will extract a wikipedia page (with OpenAI, https://en.wikipedia.org/wiki/OpenAI) to test NER with Spacy.
There are 573 entities in the page.

In [22]:
# getting wikipedia page of Open AI
wikip = wikipedia.page('OpenAI')
article = nlp(wikip.content)
len(article.ents)

1086

In [23]:
# count the number of entitie types found from wikipedia page
labels = [x.label_ for x in article.ents]
Counter(labels)

Counter({'ORG': 393,
         'NORP': 11,
         'GPE': 137,
         'PERSON': 188,
         'DATE': 212,
         'PERCENT': 10,
         'MONEY': 62,
         'CARDINAL': 44,
         'FAC': 2,
         'ORDINAL': 11,
         'WORK_OF_ART': 5,
         'PRODUCT': 7,
         'LOC': 3,
         'LAW': 1})

In [24]:
# getting the top 10 words recognised as named entity
items = [x.text for x in article.ents]
Counter(items).most_common(10)

[('OpenAI', 197),
 ('AI', 57),
 ('Microsoft', 36),
 ('Altman', 17),
 ('Sam Altman', 16),
 ('PBC', 9),
 ('Elon Musk', 9),
 ('Greg Brockman', 9),
 ('first', 8),
 ('OpenAI LP', 8)]

In [25]:
sentences = [x for x in article.sents]
print(sentences[0])

OpenAI Global, LLC is an American artificial intelligence (AI) research organization consisting of a for-profit public benefit corporation (PBC) and a nonprofit foundation, headquartered in San Francisco.


In [26]:
type(sentences[0])

spacy.tokens.span.Span

In [27]:
# display each tag of the sentence
print([(x, x.ent_iob_, x.ent_type_) for x in sentences[0]])

[(OpenAI, 'O', ''), (Global, 'O', ''), (,, 'O', ''), (LLC, 'B', 'ORG'), (is, 'O', ''), (an, 'O', ''), (American, 'B', 'NORP'), (artificial, 'O', ''), (intelligence, 'O', ''), ((, 'O', ''), (AI, 'O', ''), (), 'O', ''), (research, 'O', ''), (organization, 'O', ''), (consisting, 'O', ''), (of, 'O', ''), (a, 'O', ''), (for, 'O', ''), (-, 'O', ''), (profit, 'O', ''), (public, 'O', ''), (benefit, 'O', ''), (corporation, 'O', ''), ((, 'O', ''), (PBC, 'B', 'ORG'), (), 'O', ''), (and, 'O', ''), (a, 'O', ''), (nonprofit, 'O', ''), (foundation, 'O', ''), (,, 'O', ''), (headquartered, 'O', ''), (in, 'O', ''), (San, 'B', 'GPE'), (Francisco, 'I', 'GPE'), (., 'O', '')]


In [28]:
# display whole sentences using render()
displacy.render(nlp(str(sentences)), jupyter=True, style='ent')

In [29]:
!python -m spacy download ko_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 77.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ko_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [30]:
import ko_core_news_sm
nlp = ko_core_news_sm.load()

In [31]:
wikipedia.set_lang('ko')

# Excercise

Get a Korean wikipedia page(https://ko.wikipedia.org/wiki/오픈AI), and display its named entities using displacy.render().

In [32]:
# Please complete this